# TerraScout AI — Producto 2: Valuación Hedónica y Arbitraje ML

Notebook alineado 100% con el repositorio `Capstone/`:
- **Cero ETL duplicado:** consume `data/processed/dataset_limpio.csv` generado por `actors/apify_etl/my_actor/etl_core.py`.
- **Cero `np.random` y cero reglas hardcodeadas:** lee todos los parámetros desde `config/business_rules.json`.
- **Anti-leakage estricto:** `train_test_split` estratificado previo a `ColumnTransformer` + `Pipeline`, comparación 5-Fold CV (`DummyRegressor`, `Ridge`, `RandomForestRegressor`, `HistGradientBoostingRegressor`), arbitraje Out-of-Fold y serialización Pickle con verificación `dif_max < 1e-6`.

In [ ]:
import json
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path('.').resolve()
if not (REPO_ROOT / 'config' / 'business_rules.json').exists() and (REPO_ROOT.parent / 'config' / 'business_rules.json').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from actors.apify_ml.my_actor.ml_core import (
    format_telegram_summary,
    load_business_rules,
    load_clean_dataset,
    score_opportunities,
    train_and_evaluate_hedonic_model,
)
from src.run_producto2_pipeline import run_producto2_train

rules = load_business_rules(str(REPO_ROOT / 'config' / 'business_rules.json'))
df_clean = load_clean_dataset(str(REPO_ROOT / 'data' / 'processed' / 'dataset_limpio.csv'))
print(f'Dataset limpio oficial cargado: {len(df_clean)} filas x {df_clean.shape[1]} columnas')

In [ ]:
res = train_and_evaluate_hedonic_model(df_clean, rules)
df_comp = pd.DataFrame(res['comparison'])
display(df_comp)
print(f"Ganador por CV: {res['bundle']['modelo_nombre']} | dif_max Pickle: {res['dif_max']:.2e}")

In [ ]:
df_imp = pd.DataFrame(res['feature_importances']).head(10)
ax = df_imp.sort_values('importancia_pct').plot(kind='barh', x='feature', y='importancia_pct', legend=False, figsize=(8, 4), color='#2a9d8f')
ax.set_title('Top 10 Importancias de Variables — Modelo Hedónico Producto 2')
ax.set_xlabel('Importancia (%)')
plt.tight_layout()
plt.show()

In [ ]:
oport = res['df_oportunidades']
print(f"Oportunidades ML (Out-of-Fold + Test): {len(oport)} de {len(res['df_scored'])} lotes")
cols_vista = [
    'id_registro', 'barrio_localidad', 'precio_usd', 'superficie_m2',
    'precio_m2_usd', 'precio_modelo_m2', 'gap_arbitraje_pct',
    'pendiente_media_grados', 'riesgo_ambiental', 'flag_revision', 'split_evaluacion'
]
display(oport[cols_vista].head(15))

In [ ]:
# Persistir artefactos oficiales en models/ y data/processed/producto2/
out = run_producto2_train()

# Verificación post-carga Pickle desde disco (Clase 40)
with open(out['pkl_path'], 'rb') as f:
    loaded_bundle = pickle.load(f)

df_scored_disk, df_oport_disk = score_opportunities(df_clean.head(5), loaded_bundle, rules)
display(df_scored_disk[['id_registro', 'barrio_localidad', 'precio_m2_usd', 'precio_modelo_m2', 'gap_arbitraje_pct', 'es_oportunidad_ml']])
print(format_telegram_summary(oport, top_n=3))